In [154]:
%reset -f 
# resetting stored variables in case there's something weird cached
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys
from dataclasses import dataclass, field
import bd_warehouse.thread, bd_warehouse.fastener 
from pathlib import Path
from sympy import false
from casadi import diag
from build123d.topology.composite import Part
import path
import yaml
import string
import copy

ALL UNITS IN MM
DON'T @ ME

In [155]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 
print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")
reset_show()
startTime = time.perf_counter()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


In [156]:
# INPUTS GO HERE
# Neurotrophic Geometry/Solids Pathfinder/Originator/Generator/Optimizer? NOPE
# Neurotrophic Rhesus Optimized 
# NUTS
# Neurotrophic United/unionizing/unironical/underloft Topology Solution?
# ROOTS: Reconfigurable Originator of Translational Solids

# Chamber identity
chamberIdentity = "GoliathPosterior" # Other possibilities: GoliathAnterior, MalachiRight, MalachiLeft. More TBA
majorVersion = 1; minorVersion = 0; patchVersion = 0
softwareVersion = f"V{majorVersion}.{minorVersion}.{patchVersion}"

# Just for the moment we are presuming a universal x value, so...
# at xValue, yOffset = (2,3) we start getting issues with geometry overlapping in funny ways. Make nubs smaller?
# Same as above at (2,-4)
xValue = 1.75 # Domain: -5 to +5 mm
yOffset = 0 # Range: -3, -2, -1, 0, 1, 2, 3...
iteratorOffset = 5 #5 works, 6 works, 4 does not

"""
Original y values, 1-4: 7.7, 2.475, -1.775, -7
"""

penetration1 = (xValue, 5.00 + yOffset)
penetration2 = (xValue, -0.25 + yOffset)
penetration3 = (xValue, -4.5 + yOffset)
penetration4 = (xValue, -9.75 + yOffset)
"""
# Testing: 
# Coordinate sites / Penetration center points - these will be inputs which generate whole structure
penetration1 = (xValue, (iteratorOffset * 1.5)+ yOffset)
penetration2 = (xValue, (iteratorOffset * 0.5) + yOffset)
penetration3 = (xValue, (-iteratorOffset * 0.5) + yOffset)
penetration4 = (xValue, (-iteratorOffset * 1.5) + yOffset)
"""
points2D = (penetration1, penetration2, penetration3, penetration4)

# Diagnostic mode yes or no? If y, then we should set it up to have Show() commands for diagnostic purposes that turn on with a time.sleep() so we can do analysis and show off what's happening. 
diagnosticMode = 3 # user input field eventually. 0 is no, 1 is full diagnostics, 2 is timefield reporting for subfield functions only. 
smallDiagnosticTime = 0.1
largeDiagnosticTime = 0.5

# Do you want to save this file? 0 is no, 1 is yes. Needs a file name; this will be used as a suffix. 
fileName = f"ParametricGuideTubeFrame{softwareVersion}"
saveyn = 1

In [157]:
# Below are some parameters which govern the GT shaft dimensions. Easy permutation layer for later designs if needed.
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
innerShaftDiameter = 0.57
zOffset = 2

# These should probably be divined algorithmically but right now they're just the victims of trial and error on my part
handPickedIndicies = [1,5,9,13]

In [158]:
# Class-based data storage 

@dataclass
class Config:
    chamber: str = None
    file = None
    largeDiagnosticTime = largeDiagnosticTime
    smallDiagnosticTime = smallDiagnosticTime
    diagnostics: int = None


@dataclass
class Scene:
    stages: list = field(default_factory = list)
    coordinates: tuple = None
    basisCylinder: Part = None
    sites: list = field(default_factory = list)
    shaftOD: object = 4
    shaftID: object = 0.57
    shaftHeight = 3.85
    shaftZOffset = 2
    
sites = []
planeHeightIndicator = 0

@dataclass
class Site:
    shaft: Part = None
    startingPlane: Plane = None
    bottomSolid: Part = None
    nubs: list = None
    arms: list = field(default_factory = list)
    combined: object = None
    coordinates: tuple = None


In [159]:
# Initializing the scene and config

config = Config(
    chamber = chamberIdentity,
    diagnostics = diagnosticMode
)

scene = Scene(
    stages = [],
    coordinates = points2D,
    sites = []
)

for i, coordinate in enumerate(scene.coordinates):
    scene.sites.append(Site(
        coordinates = coordinate
    ))

In [160]:
# Scene wrapper for basisCylinder

def basisCylinderConstructor(scene, config):

    scene.stages.append(copy.deepcopy(scene))

    # Calling chamberBasisConstructor to construct the basisCylinder
    scene.basisCylinder = chamberBasisConstructor(config.chamber, config.diagnostics)
    # note: 10% of time

    if config.diagnostics == 3:
        print(scene.stages[-1])
        print(scene)

    return(scene, config)

scene, config = basisCylinderConstructor(scene, config)


Scene(stages=[], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=None, sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, 5.0)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -0.25)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -4.5)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -9.75))], shaftOD=4, shaftID=0.57)
Scene(stages=[Scene(stages=[], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=None, sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, 5.0)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -0.25)), Site(shaft=None, startingPl

Copy pasta for conversion: scene, config.chamber, scene.coordinates, shaftHeight = scene.shaftHeight, diagnosticMode = scene.diagnostics, shaftDiameter = scene.shaftOD, largeDiagnosticTime = config.largeDiagnosticTime, smallDiagnosticTime=config.smallDiagnosticTime

In [161]:
# Scene wrapper for meshPlanes

def meshPlanesScene(scene, config):

    scene.stages.append(copy.deepcopy(scene))

    # Calling meshPlanes to get our starting extrusion planes, ending extrusion solids, and 3D coordinates for each site
    points3D, bottomSurfaceSolids, startingOffsetPlanes = meshPlanes(config.chamber, scene.coordinates, shaftHeight = scene.shaftHeight, diagnosticMode = config.diagnostics, shaftDiameter = scene.shaftOD, largeDiagnosticTime = config.largeDiagnosticTime, smallDiagnosticTime=config.smallDiagnosticTime)

    # Coordinates are transferred to sites
    for i, x in enumerate(scene.sites):
        x.coordinates = points3D[i]

    # Coordinates are transferred to the scene itself
    scene.coordinates = points3D

    # bottomSurfaceSolids are transferred to the scene.
    scene.bottomSurfaceSolids = bottomSurfaceSolids

    # startingOffsetPlanes are transferred to the scene
    scene.startingOffsetPlanes = startingOffsetPlanes

    if config.diagnostics == 3:
        print(scene.stages[-1])
        print(scene)

    return(scene, config)

scene, config = meshPlanesScene(scene, config)


# config.chamber, scene.coordinates, shaftHeight = scene.shaftHeight, diagnosticMode = config.diagnostics, shaftDiameter = scene.shaftOD, largeDiagnosticTime = config.largeDiagnosticTime, smallDiagnosticTime=config.smallDiagnosticTime

Scene(stages=[Scene(stages=[], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=None, sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, 5.0)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -0.25)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -4.5)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -9.75))], shaftOD=4, shaftID=0.57)], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=Part at 0x23f475ce2a0, label(), #children(0), sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, 5.0)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -0.25)), Si

In [162]:
def shaftsScene(scene, config):

    scene.stages.append(copy.deepcopy(scene))

    # Calling shaftConstructor to construct shafts at each site, storing within the scene
    scene.shafts = shaftConstructor(startingOffsetPlanes = scene.startingOffsetPlanes, bottomSurfaceSolids = scene.bottomSurfaceSolids, innerShaftDiameter = scene.shaftID, shaftDiameter = scene.shaftOD, diagnosticMode = config.diagnostics, largeDiagnosticTime = config.largeDiagnosticTime, smallDiagnosticTime = config.smallDiagnosticTime)

    print(scene.shafts)
    # Transferring to sites
    for i, x in enumerate(scene.sites):
        scene.sites[i].shaft = scene.shafts[i]

    if config.diagnostics == 3:
        print(scene.stages[-1])
        print(scene)

    return(scene, config)

scene, config = shaftsScene(scene, config)



[Part at 0x23f476a2e70, label(), #children(0), Part at 0x23f4775fb60, label(), #children(0), Part at 0x23f4775fe30, label(), #children(0), Part at 0x23f4775e720, label(), #children(0)]
Scene(stages=[Scene(stages=[], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=None, sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, 5.0)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -0.25)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -4.5)), Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=None, arms=[], combined=None, coordinates=(1.75, -9.75))], shaftOD=4, shaftID=0.57), Scene(stages=[Scene(stages=[], coordinates=((1.75, 5.0), (1.75, -0.25), (1.75, -4.5), (1.75, -9.75)), basisCylinder=None, sites=[Site(shaft=None, startingPlane=None, bottomSolid=None, nubs=N

In [163]:
# Constructing nubs for each site
nubsList, nubTemplatesFlattened = nubConstructor(startingOffsetPlanes, diagnosticMode = diagnosticMode, largeDiagnosticTime = largeDiagnosticTime, smallDiagnosticTime = smallDiagnosticTime)

NameError: name 'startingOffsetPlanes' is not defined

In [ ]:
# Constructing intermediate geometry with lofts between sites
prunedParts, lofts, overlapSolids = loftConstructor(nubsList, nubTemplatesFlattened, startingOffsetPlanes, diagnosticMode = diagnosticMode)

In [ ]:
# Generating positional arms to connect nubs and basis cylinder
outputArmStream, testArms = armStreamConstructor(startingOffsetPlanes,nubsList,handPickedIndicies, diagnosticMode = diagnosticMode)

In [ ]:
# Constructing initial sites, storing previously gathered information in them
sites = siteConstructor(Site, startingOffsetPlanes,shaftListWithThroughHolesFilletedTwice,nubsList,prunedParts,outputArmStream, diagnosticMode = diagnosticMode)

In [ ]:
# Construct guide tube frame from all the parts we've just made
guideTubeFrame = guideTubeFrameConstructor(sites, basisCylinder, nubsList, prunedParts, lofts, overlapSolids, diagnosticMode = diagnosticMode)

NameError: name 'basisCylinder' is not defined

In [ ]:
# Saving file as STL if saveyn = 1, and printing the file name
finalName = saveMe(points2D,startTime,guideTubeFrame,saveyn,fileName,chamberIdentity, diagnosticMode = diagnosticMode)